In [2]:
import pandas as pd

import os
import s3fs


fs = s3fs.S3FileSystem(
    client_kwargs={'endpoint_url': 'https://'+'minio.lab.sspcloud.fr'},
    key = os.environ["AWS_ACCESS_KEY_ID"], 
    secret = os.environ["AWS_SECRET_ACCESS_KEY"], 
    token = os.environ["AWS_SESSION_TOKEN"])

"""
with fs.open("s3://lab/art_net_dec.parquet") as f : 
    dfa = pd.read_parquet(f)



with fs.open("s3://lab/conf_net_dec.parquet") as f : 
    dfc = pd.read_parquet(f)"""

#with fs.open("s3://lab/mem/full_name_list.csv") as f : 
   # nl = pd.read_csv(f, index_col=[0])
"""
with fs.open("s3://lab/mem/n_to_g.csv") as f : 
    ntg = pd.read_csv(f, index_col=[0])"""
"""
with fs.open("s3://lab/kgnd.csv") as f:
    kgnd = pd.read_csv(f)
with fs.open("s3://lab/ignd.csv") as f:
    ignd = pd.read_csv(f)
with fs.open("s3://lab/wgnd.csv") as f:
    wgnd = pd.read_csv(f)
with fs.open("s3://lab/jgnd.csv") as f:
    jgnd = pd.read_csv(f)
with fs.open("s3://lab/cgnd.csv") as f:
    cgnd = pd.read_csv(f)
with fs.open("s3://lab/usgnd.csv") as f: 
    usgnd = pd.read_csv(f)"""

'\nwith fs.open("s3://lab/kgnd.csv") as f:\n    kgnd = pd.read_csv(f)\nwith fs.open("s3://lab/ignd.csv") as f:\n    ignd = pd.read_csv(f)\nwith fs.open("s3://lab/wgnd.csv") as f:\n    wgnd = pd.read_csv(f)\nwith fs.open("s3://lab/jgnd.csv") as f:\n    jgnd = pd.read_csv(f)\nwith fs.open("s3://lab/cgnd.csv") as f:\n    cgnd = pd.read_csv(f)\nwith fs.open("s3://lab/usgnd.csv") as f: \n    usgnd = pd.read_csv(f)'

GND preprocessing

In [10]:
gnd=pd.concat([wgnd,cgnd,kgnd,ignd,jgnd,usgnd],axis=0,ignore_index=True)
gnd=gnd.drop(columns=["Count"])
gnd["name"] = gnd["Name"].str.lower()
gnd = gnd.drop("Name",axis=1)
#gnd = gnd.drop_duplicates()
gnd

,Gender,name
0,F,baby
1,F,aisyah
2,F,anela
3,F,anela
4,F,fiyinfoluwa
...,...,...
27351475,M,zylyn
27351476,M,zymiere
27351477,M,zypher
27351478,M,zyre


In [11]:
import re, string

punct = re.escape(string.punctuation)
gnd["name"] = gnd["name"].str.replace(f'[{punct}]', '', regex=True)

In [12]:
# Count occurrences of each Name/Gender pair
counts = gnd.groupby(["name", "Gender"]).size().reset_index(name="n")

# Sort so M comes first (tiebreak), then by count descending
counts = counts.sort_values(["name", "n", "Gender"], ascending=[True, False, False])

# Keep first = highest count, M wins ties
gnd = counts.drop_duplicates(subset="name")[["name", "Gender"]]
gnd

,name,Gender
1,a,M
2,aa,F
3,aaa,M
4,aaaa,M
5,aaabkhan,M
...,...,...
2887624,ㄌ,M
2887625,ㄎ,M
2887626,㈩,M
2887627,凉凉,M


In [27]:
#duplicates
pd.concat(g for _, g in ntg.groupby("name") if len(g) > 1)

,name,Gender
5877,a,M
15833,a,M
23388,a,M
24930,a,M
40150,a,M
...,...,...
237895,zuying,M
27568,zuyuan,M
214133,zuyuan,M
53529,zy,M


NL prepro

In [8]:
dfa["authors"] = dfa["authors"].apply(lambda lst: [d["name"] for d in lst])
dfc["authors"] = dfc["authors"].apply(lambda lst: [d["name"] for d in lst])

In [16]:
nla = dfa["authors"].explode().str.split(' ').str[0]
nlc = dfc["authors"].explode().str.split(' ').str[0]

In [19]:
nl = pd.concat([nla,nlc]).drop_duplicates()

In [17]:
nl = nl[~nl["authors"].str.contains(r"^[A-Za-z]\.$", na=False)]
nl = nl[~nl["authors"].str.contains(r"^[A-Za-z]\.-[A-Za-z]\.$", na=False)]
nl = nl[~nl["authors"].str.contains(r"^[A-Za-z]-[A-Za-z]\.$", na=False)]
nl = nl[~nl["authors"].str.contains(r"^[A-Za-z][A-Za-z]\.$", na=False)]
nl = nl[~nl["authors"].str.contains(r"^[A-Za-z][A-Za-z][A-Za-z]\.$", na=False)]

In [18]:
# for jean-marie
nl = nl["authors"].str.replace("-", " ").str.split(" ").str[0].str.lower()

In [19]:
nl.to_csv("s3://lab/mem/full_name_list.csv")

NTG prepro

In [20]:
with fs.open("s3://lab/mem/full_name_list.csv") as f : 
    nl = pd.read_csv(f, index_col=[0])
nl

,authors
0,kai
1,richard
1,stefan
2,herbert
3,sophie
...,...
3749381,siwaphon
3749388,tasi
3749413,xiangxu
3749415,pradhyumansinh


In [31]:
ntg = pd.DataFrame(nl).drop_duplicates().merge(gnd,how="left",left_on="authors",right_on="name")
ntg = ntg.drop("authors",axis=1)
ntg = ntg[~ntg["name"].isna()]
ntg

,name,Gender
0,kai,M
1,richard,M
2,stefan,M
3,herbert,M
4,sophie,F
...,...,...
231276,mincao,M
231278,naiguo,M
231284,tiaoyun,M
231297,tasi,F


In [30]:
gnd#.drop_duplicates(subset="name")

,name,Gender
1,a,M
2,aa,F
3,aaa,M
4,aaaa,M
5,aaabkhan,M
...,...,...
2887624,ㄌ,M
2887625,ㄎ,M
2887626,㈩,M
2887627,凉凉,M


In [53]:
!pip install gender-guesser

In [71]:
import gender_guesser.detector as gg

d = gg.Detector()
ng_dict = pd.DataFrame()
ng_dict["name"] = nl
ng_dict["gender"] = nl.apply(lambda x: d.get_gender(x))
ng_dict

,name,gender
0,Kai,male
1,Richard,male
1,Stefan,male
2,Herbert,male
3,Sophie,female
...,...,...
3749381,Siwaphon,unknown
3749388,Tasi-Ying,unknown
3749413,XiangXu,unknown
3749415,Pradhyumansinh,unknown


In [72]:
ng_dict[ng_dict["gender"]=="unknown"]

,name,gender
4,NaN,unknown
27,Hans-Jürgen,unknown
42,Nicolaos,unknown
87,Brikenrikena,unknown
96,Catalin,unknown
...,...,...
3749381,Siwaphon,unknown
3749388,Tasi-Ying,unknown
3749413,XiangXu,unknown
3749415,Pradhyumansinh,unknown


In [32]:
ntg.to_csv("s3://lab/mem/n_to_g.csv")

art net prepro

In [34]:
dfa["authors"] = dfa["authors"].apply(lambda lst: [d["name"] for d in lst])
attr = dfa.explode("authors")
#attr["name"] = attr["authors"].str.split(" ").str[0].str.lower()
#attr = attr.merge(ntg[["authors","Gender"]],how="left",left_on="name",right_on="authors")
attr

,title,journal,year,authors,dblp_uri,doi
0,Auswirkung der Digitalisierung auf die Systeml...,Elektrotech. Informationstechnik,2019,Kai Schlabitz,journals/ei/Schlabitz19,https://doi.org/10.1007/s00502-019-0687-y
1,EMF-Personenschutz: Neue Aspekte in der numeri...,Elektrotech. Informationstechnik,2020,Richard Überbacher,journals/ei/UberbacherC20,https://doi.org/10.1007/s00502-020-00791-z
1,EMF-Personenschutz: Neue Aspekte in der numeri...,Elektrotech. Informationstechnik,2020,Stefan Cecil,journals/ei/UberbacherC20,https://doi.org/10.1007/s00502-020-00791-z
2,Zur Genesis der Forschungsstelle für Integrier...,Elektrotech. Informationstechnik,2022,Herbert Mang,journals/ei/Mang22,https://doi.org/10.1007/s00502-022-01049-6
3,100 % erneuerbare Energie für Österreichs Indu...,Elektrotech. Informationstechnik,2021,Sophie Knöttner,journals/ei/KnottnerGDD21,https://doi.org/10.1007/s00502-021-00953-7
...,...,...,...,...,...,...
3992220,Analysis of projected hydrological behavior of...,Hydrology and Earth System Sciences,2012,Rita Ley,persons/CasperGGGHLR12,https://www.wikidata.org/entity/Q114958217
3992220,Analysis of projected hydrological behavior of...,Hydrology and Earth System Sciences,2012,Andreas Rock,persons/CasperGGGHLR12,https://www.wikidata.org/entity/Q114958217
3992221,Common Subexpression Identification in General...,"Technical Rep. UKSC 0060, IBM United Kingdom S...",1974,Patrick A. V. Hall,persons/Hall74,None
3992222,Interactive Support for Non-Programmers: The R...,"Research Report / RJ / IBM / San Jose, California",1974,E. F. Codd,persons/CoddD74,None


In [35]:
attr = attr.groupby("authors", as_index=False).agg({col: list for col in attr.columns if col != "authors"})
attr

In [36]:
attr

,authors,title,journal,year,dblp_uri,doi
0,'Damola Adediji,"[Undermining competition, undermining markets?...",[Big Data Soc.],[2025],[journals/bigdatasociety/BirchA25],[None]
1,'Maseka Lesaoana,[A transportation branch and bound algorithm f...,"[Int. J. Syst. Assur. Eng. Manag., Oper. Res.]","[2015, 2001]","[journals/saem/MunapoLNK15, journals/ior/HallL...","[https://doi.org/10.1007/s13198-015-0343-9, ht..."
2,'Osaiasi Lolohea,[Visual literacy shown through a magnifying le...,[Interact. Technol. Smart Educ.],[2023],[journals/itse/ReddySCLT23],[https://doi.org/10.1108/ITSE-01-2022-0007]
3,(David) Jing Dai,[AutonomousGIS 2017 workshop report: the First...,[ACM SIGSPATIAL Special],[2017],[journals/sigspatial/GaoDHXC17],[https://doi.org/10.1145/3178392.3178407]
4,(Max) Zong-Ming Cheng,[MicroSyn: A user friendly tool for detection ...,"[BMC Bioinform., BMC Bioinform.]","[2011, 2009]","[journals/bmcbi/CaiYTC11, journals/bmcbi/XuZHZ...","[https://www.wikidata.org/entity/Q28741346, ht..."
...,...,...,...,...,...,...
2277316,Þorsteinn Sæmundsson,[Inferring 2D Local Surface-Deformation Veloci...,[Remote. Sens.],[2022],[journals/remotesensing/DittrichHTS22],[None]
2277317,Þröstur Pétursson,[Bone Mineral Density and Fracture Risk Assess...,[Comput. Math. Methods Medicine],[2015],[journals/cmmm/PeturssonEGMMHJ15],[https://www.wikidata.org/entity/Q36055821]
2277318,Þórhildur Þorleiksdóttir,[Exquisitor: Interactive Learning at Large.],[CoRR],[2019],[journals/corr/abs-1904-08689],[None]
2277319,Þórir Hrafn Harðarson,[Aligning Language Models for Icelandic Legal ...,[CoRR],[2025],[journals/corr/abs-2504-18180],[None]


In [40]:
import re, string

punct = re.escape(string.punctuation)
attr["authors"] = attr["authors"].str.replace(f'[{punct}]', '', regex=True)
attr["name"] = attr["authors"].str.split(" ").str[0].str.lower()
attr["name"] = attr["name"].str.replace("-", " ").str.split(" ").str[0]
#attr = attr.merge(ntg[["Name","Gender"]],how="left",left_on="name",right_on="Name")
#attr = attr.drop(df.columns["Name"], axis = 1)
attr

,authors,title,journal,year,dblp_uri,doi,name,Gender
0,Damola Adediji,"[Undermining competition, undermining markets?...",[Big Data Soc.],[2025],[journals/bigdatasociety/BirchA25],[None],damola,M
1,Maseka Lesaoana,[A transportation branch and bound algorithm f...,"[Int. J. Syst. Assur. Eng. Manag., Oper. Res.]","[2015, 2001]","[journals/saem/MunapoLNK15, journals/ior/HallL...","[https://doi.org/10.1007/s13198-015-0343-9, ht...",maseka,NaN
2,Osaiasi Lolohea,[Visual literacy shown through a magnifying le...,[Interact. Technol. Smart Educ.],[2023],[journals/itse/ReddySCLT23],[https://doi.org/10.1108/ITSE-01-2022-0007],osaiasi,NaN
3,David Jing Dai,[AutonomousGIS 2017 workshop report: the First...,[ACM SIGSPATIAL Special],[2017],[journals/sigspatial/GaoDHXC17],[https://doi.org/10.1145/3178392.3178407],david,NaN
4,Max ZongMing Cheng,[MicroSyn: A user friendly tool for detection ...,"[BMC Bioinform., BMC Bioinform.]","[2011, 2009]","[journals/bmcbi/CaiYTC11, journals/bmcbi/XuZHZ...","[https://www.wikidata.org/entity/Q28741346, ht...",max,NaN
...,...,...,...,...,...,...,...,...
2277316,Þorsteinn Sæmundsson,[Inferring 2D Local Surface-Deformation Veloci...,[Remote. Sens.],[2022],[journals/remotesensing/DittrichHTS22],[None],þorsteinn,NaN
2277317,Þröstur Pétursson,[Bone Mineral Density and Fracture Risk Assess...,[Comput. Math. Methods Medicine],[2015],[journals/cmmm/PeturssonEGMMHJ15],[https://www.wikidata.org/entity/Q36055821],þröstur,NaN
2277318,Þórhildur Þorleiksdóttir,[Exquisitor: Interactive Learning at Large.],[CoRR],[2019],[journals/corr/abs-1904-08689],[None],þórhildur,NaN
2277319,Þórir Hrafn Harðarson,[Aligning Language Models for Icelandic Legal ...,[CoRR],[2025],[journals/corr/abs-2504-18180],[None],þórir,NaN


In [43]:
attr = attr.merge(ntg,how="left",on="name")
#attr = attr.drop(df.columns["Name"], axis = 1)
attr

,authors,title,journal,year,dblp_uri,doi,name,Gender
0,Damola Adediji,"[Undermining competition, undermining markets?...",[Big Data Soc.],[2025],[journals/bigdatasociety/BirchA25],[None],damola,M
1,Maseka Lesaoana,[A transportation branch and bound algorithm f...,"[Int. J. Syst. Assur. Eng. Manag., Oper. Res.]","[2015, 2001]","[journals/saem/MunapoLNK15, journals/ior/HallL...","[https://doi.org/10.1007/s13198-015-0343-9, ht...",maseka,NaN
2,Osaiasi Lolohea,[Visual literacy shown through a magnifying le...,[Interact. Technol. Smart Educ.],[2023],[journals/itse/ReddySCLT23],[https://doi.org/10.1108/ITSE-01-2022-0007],osaiasi,NaN
3,David Jing Dai,[AutonomousGIS 2017 workshop report: the First...,[ACM SIGSPATIAL Special],[2017],[journals/sigspatial/GaoDHXC17],[https://doi.org/10.1145/3178392.3178407],david,M
4,Max ZongMing Cheng,[MicroSyn: A user friendly tool for detection ...,"[BMC Bioinform., BMC Bioinform.]","[2011, 2009]","[journals/bmcbi/CaiYTC11, journals/bmcbi/XuZHZ...","[https://www.wikidata.org/entity/Q28741346, ht...",max,M
...,...,...,...,...,...,...,...,...
2277316,Þorsteinn Sæmundsson,[Inferring 2D Local Surface-Deformation Veloci...,[Remote. Sens.],[2022],[journals/remotesensing/DittrichHTS22],[None],þorsteinn,NaN
2277317,Þröstur Pétursson,[Bone Mineral Density and Fracture Risk Assess...,[Comput. Math. Methods Medicine],[2015],[journals/cmmm/PeturssonEGMMHJ15],[https://www.wikidata.org/entity/Q36055821],þröstur,NaN
2277318,Þórhildur Þorleiksdóttir,[Exquisitor: Interactive Learning at Large.],[CoRR],[2019],[journals/corr/abs-1904-08689],[None],þórhildur,NaN
2277319,Þórir Hrafn Harðarson,[Aligning Language Models for Icelandic Legal ...,[CoRR],[2025],[journals/corr/abs-2504-18180],[None],þórir,NaN


In [ ]:
# NA
len(attr[attr["Gender"].isna()])/len(attr)

0.09237476842307255

In [3]:
with fs.open("s3://lab/art_net_dec.parquet") as f : 
    df = pd.read_parquet(f)

df

,title,journal,year,authors,dblp_uri,doi
0,Auswirkung der Digitalisierung auf die Systeml...,Elektrotech. Informationstechnik,2019,"[{'name': 'Kai Schlabitz', 'orcid': None}]",journals/ei/Schlabitz19,https://doi.org/10.1007/s00502-019-0687-y
1,EMF-Personenschutz: Neue Aspekte in der numeri...,Elektrotech. Informationstechnik,2020,"[{'name': 'Richard Überbacher', 'orcid': None}...",journals/ei/UberbacherC20,https://doi.org/10.1007/s00502-020-00791-z
2,Zur Genesis der Forschungsstelle für Integrier...,Elektrotech. Informationstechnik,2022,"[{'name': 'Herbert Mang', 'orcid': None}]",journals/ei/Mang22,https://doi.org/10.1007/s00502-022-01049-6
3,100 % erneuerbare Energie für Österreichs Indu...,Elektrotech. Informationstechnik,2021,"[{'name': 'Sophie Knöttner', 'orcid': None}, {...",journals/ei/KnottnerGDD21,https://doi.org/10.1007/s00502-021-00953-7
4,Application of multilateration for microphone ...,Elektrotech. Informationstechnik,2021,[],journals/ei/WimbergerR21,https://doi.org/10.1007/s00502-021-00885-2
...,...,...,...,...,...,...
3992218,"Derivability, Redundancy and Consistency of Re...","Research Report / RJ / IBM / San Jose, California",1969,"[{'name': 'E. F. Codd', 'orcid': None}]",persons/Codd69,None
3992219,Normalized Data Base Structure: A Brief Tutorial.,"Research Report / RJ / IBM / San Jose, California",1971,"[{'name': 'E. F. Codd', 'orcid': None}]",persons/Codd71b,None
3992220,Analysis of projected hydrological behavior of...,Hydrology and Earth System Sciences,2012,"[{'name': 'Gayane Grigoryan', 'orcid': None}, ...",persons/CasperGGGHLR12,https://www.wikidata.org/entity/Q114958217
3992221,Common Subexpression Identification in General...,"Technical Rep. UKSC 0060, IBM United Kingdom S...",1974,"[{'name': 'Patrick A. V. Hall', 'orcid': None}]",persons/Hall74,None


In [45]:
attr.to_parquet("s3://lab/mem/art_attr.parquet")

In [52]:
sample = attr["name"].drop_duplicates().sample(1000)

In [54]:
sample.to_csv("s3://lab/mem/name_sample.csv")

In [4]:
df = df["authors"].apply(lambda lst: [d["name"] for d in lst])

In [9]:
pd.DataFrame(df).to_parquet("s3://lab/mem/art_edges.parquet")

Network modelisation

In [3]:
with fs.open("s3://lab/mem/art_edges.parquet") as f : 
    edges = pd.read_parquet(f)

edges

,authors
0,[Kai Schlabitz]
1,"[Richard Überbacher, Stefan Cecil]"
2,[Herbert Mang]
3,"[Sophie Knöttner, Roman Geyer, Christian Diend..."
4,[]
...,...
3992218,[E. F. Codd]
3992219,[E. F. Codd]
3992220,"[Gayane Grigoryan, Oliver Gronz, Oliver Gutjah..."
3992221,[Patrick A. V. Hall]


In [4]:
%pip install networkx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 74.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [11]:
import networkx as nx
from itertools import combinations



G = nx.Graph()
for team in edges["authors"]:
    G.add_edges_from(combinations(team, 2))

In [37]:
G.remove_edges_from(nx.selfloop_edges(G))

In [38]:
graphs = list(nx.connected_components(G))

3 graphs à plus de 50 membres → on retient que le premier graph et les autres sont considérés comme des teams

In [39]:
big_graphs = []

for i in graphs : 
    if len(i)>50 : 
        big_graphs.append(i)
len(big_graphs)

3

In [ ]:
core = G.subgraph(graphs[0]).copy()

stat desc

In [36]:
nx.rich_club_coefficient(core, normalized=False, seed=42)

Exception: rich_club_coefficient is not implemented for graphs with self loops.